In [1]:
import os
os.sys.path.append('/data/scratch/globc/bonassies/workspace/swot_for_flood')

import configparser
from pathlib import Path
from core.swot_project import SwotProject
from core.swot_raster import SwotRaster

main_path = "/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D"

# Download and rasterize notebook

This notebook explain how to use the swot_for_flood library to download and rasterize the SWOT data for a given region and time period. 

## Define a project

This library is designed to work with a project. A project is a directory that contains the configuration file `config.cfg` and the data. The configuration file contains the parameters of the project.

In this notebook, we will create a project named "example_download_rasterize" in the directory "examples". The project will contain the SWOT data for the region of interest and time period defined in the configuration file.

In [2]:
config = configparser.ConfigParser()
config.read(main_path + '/config.cfg')

print(type(config),dict(config['CONFIG']))

<class 'configparser.ConfigParser'> {'project': 'Chinon_T2D', 'workspace': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples', 'data_path': '/data/scratch/globc/bonassies/data', 'aoi': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/AOI_CHINON.gpkg', 'floodmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/AOI_CHINON.gpkg', 'controlmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/AOI_CHINON.gpkg', 'esa_worldcover_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/WC_cut_merged_epsg2154_nrow5231_ncol6495.tif', 'crs': '2154', 'aoi_crs': '2154', 'first_time': '2022-01-01', 'last_time': '2026-06-01', 'nodes': '8', 'download_type': 'PIXC', 'passes': '[113]', 'tile_names_selection': '[[113_237R, 113_238L, 113_238R, 113_237L]]', 'list_dry_dates': '[2024-08-03, 2024-08-24, 2024-09-14, 2024-10-05]', 'list_flood

The config can also be a dictionary

In [4]:
# from pathlib import Path

# param_dict = {
#     'project': 'Greece_EMSR692',
#     'workspace': Path("/data/scratch/globc/bonassies/workspace"),
#     'data_path': Path("/data/scratch/globc/bonassies/data"),
#     'CRS': 'EPSG:32634',
#     'first_time': "2023-09-05",
#     'last_time': "2023-09-20",
#     'aoi': Path("/data/scratch/globc/bonassies/workspace").joinpath('Greece_EMSR692',"aux_data","EMSR692_aois_V2.gpkg"),
#     'aoi_crs': 'EPSG:4326',
#     'passes': [402, 417],
#     'nodes': 8,
#     'do_download': False,
#     'download_type': 'PIXC',
#     'GDAL_NUM_THREADS': 10,
#     'GDAL_CACHEMAX': 160000,
#     'do_make_gpkg': False,
#     'do_make_tiff': False    
# }

Once the config file loaded, we can use it to create a project object. The project object will contain the parameters of the project and the data.

In [3]:
# config["CONFIG"]['do_make_gpkg'] = 'True'
# config["CONFIG"]['do_make_tiff'] = 'True'
swot_project = SwotProject(config)

# print(swot_project)


Data path already exists in /data/scratch/globc/bonassies/data or download is set to False
SWOT data already exists in /data/scratch/globc/bonassies/data/SWOT or download is set to False
SWOT project already exists in /data/scratch/globc/bonassies/data/SWOT/Chinon_T2D or download is set to False
Geopackage already exists in /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/gpkg_combined or make_gpkg is set to False
TIFF path already exists in /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters or make_tiff is set to False
No automatic download, please use the Downloader object to download the data
No pixc files found, please check the SWOT_PATH of download the data


On the first time, you should get an error because the data is not downloaded yet. The project object will download the data and store it in the project directory only if do_download is set to True in the configuration file.

You can manually download the data by calling the download method of the project object.

First we need to search for the data we want to download. We can use the search method of the project object to search for the data. The search method will return a list of the data that match the search criteria.

In [6]:
swot_project.Downloader.search_PIXC(True)
# swot_project.Downloader.search_PIXCVec(True)
# swot_project.Downloader.search_Nodes(True)

Found 745 granules
Found 445 granules within [113] passes
Found 148 granules within only studied dates


Then we can download the data by calling the download method of the project object. The download method will download the data and store it in the project directory.

In [ ]:
swot_project.Downloader.download_pool() # if multithreading download
# swot_project.Downloader.download_granules() # if single thread download


Finally we can rasterize the data by calling the rasterize method of the project object. The rasterize method will rasterize the data and store it in the project directory.

First we create gpgk file that combine the netcdf files and then we rasterize the data.

In [5]:
swot_project.Rasterizer.SWOT_PATH.name
list_tile = [name.name for name in swot_project.Rasterizer.SWOT_PATH.glob(f'*PIXC*.nc')]
tiles = [name.split('_')[5] + "_" + name.split('_')[6] for name in list_tile]
tiles = list(set(tiles))
passes = list(set([name.split('_')[0] for name in tiles]))
passes = ['113']
tiles

['113_238L', '113_237R', '113_237L', '113_238R']

In [6]:
swot_project.Rasterizer.tile_names_selection = []
for pass_num in passes:
    sublist_tile = [name for name in tiles if pass_num in name]
    swot_project.Rasterizer.tile_names_selection.append(sublist_tile)
    
swot_project.Rasterizer.tile_names_selection

[['113_238L', '113_237R', '113_237L', '113_238R']]

In [7]:
swot_project.Rasterizer.make_space = False
swot_project.Rasterizer.find_pixc() #True : only pre-treat the ganules on the studied dates (based on dry and flooded dates lists)
swot_project.Rasterizer.pixc_to_gpkg() #True : only pre-treat the ganules on the studied dates (based on dry and flooded dates lists)

>>> Converting the SWOT PIXC netcdf to geopackage
[[PosixPath('/data/scratch/globc/bonassies/data/SWOT/Chinon_T2D/SWOT_L2_HR_PIXC_033_113_238R_20250522T221758_20250522T221809_PID0_02.nc')], [PosixPath('/data/scratch/globc/bonassies/data/SWOT/Chinon_T2D/SWOT_L2_HR_PIXC_034_113_238R_20250612T190303_20250612T190314_PID0_01.nc')], [PosixPath('/data/scratch/globc/bonassies/data/SWOT/Chinon_T2D/SWOT_L2_HR_PIXC_046_113_238R_20260218T040357_20260218T040409_PID0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Chinon_T2D/SWOT_L2_HR_PIXC_046_113_238L_20260218T040357_20260218T040409_PID0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Chinon_T2D/SWOT_L2_HR_PIXC_046_113_237L_20260218T040347_20260218T040359_PID0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Chinon_T2D/SWOT_L2_HR_PIXC_046_113_237R_20260218T040347_20260218T040359_PID0_01.nc')]]
>>> Working on :
	 SWOT_L2_HR_PIXC_033_113_238R_20250522T221758_20250522T221809_PID0_02.nc
Specular Ringing: (array([Fals

/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: divide by zero encountered in divide
  result_data = func(*input_data)


>>> Working on :
	 SWOT_L2_HR_PIXC_034_113_238R_20250612T190303_20250612T190314_PID0_01.nc
Specular Ringing: (array([False,  True]), array([12691010,   303218]))
Low Coherent: (array([False,  True]), array([9858813, 3135415]))
Pixel discarded: (array([False,  True]), array([12762702,   231526]))
Bright Land Pixels discarded: (array([False,  True]), array([12979439,    14789]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: divide by zero encountered in divide
  result_data = func(*input_data)


>>> Working on :
	 SWOT_L2_HR_PIXC_046_113_238R_20260218T040357_20260218T040409_PID0_01.nc
Specular Ringing: (array([False,  True]), array([12674170,   317952]))
Low Coherent: (array([False,  True]), array([10339526,  2652596]))
Pixel discarded: (array([False,  True]), array([12763359,   228763]))
Bright Land Pixels discarded: (array([False,  True]), array([12973994,    18128]))


...

/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


	 SWOT_L2_HR_PIXC_046_113_237L_20260218T040347_20260218T040359_PID0_01.nc
Specular Ringing: (array([False,  True]), array([12971676,    76397]))
Low Coherent: (array([False,  True]), array([11064059,  1984014]))
Pixel discarded: (array([False,  True]), array([12992400,    55673]))
Bright Land Pixels discarded: (array([False,  True]), array([13046504,     1569]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


	 SWOT_L2_HR_PIXC_046_113_237R_20260218T040347_20260218T040359_PID0_01.nc
Specular Ringing: (array([False,  True]), array([13020323,    48138]))
Low Coherent: (array([False,  True]), array([10782998,  2285463]))
Pixel discarded: (array([False,  True]), array([13038076,    30385]))
Bright Land Pixels discarded: (array([False,  True]), array([13061665,     6796]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


In [ ]:
# swot_project.Rasterizer.pixc_to_gpkg()

In [8]:
swot_project.Rasterizer.gpkg_to_tiff(
    # power=2,
    # smoothing=1,
    # radius= 50,
    # max_points=20, 
    )

>>> Converting the SWOT geopackage to tiff
>>> Generate tiff for every variables
Working on  /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/gpkg_combined/SWOT_epsg2154_20260218T040347.gpkg
>>> Generate tiff for  sig0
gdal_grid -a invdistnn:power=2:smoothing=1:radius=50:max_points=20:nodata=-9999 -txe 461269.39122028934 526219.0386155185 -tye 6700870.222726153 6648566.855881953 -outsize 6495.0 5231.0 -zfield sig0 -of GTiff -ot Float32 /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/gpkg_combined/SWOT_epsg2154_20260218T040347.gpkg /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/sig0/SWOT_epsg2154_20260218T040347_sig0.tif --config GDAL_NUM_THREADS 10 --config GDAL_CACHEMAX 160000
Grid data type is "Float32"
Grid size = (6495 5231).
Corner coordinates = (461269.391220 6700870.222726)-(526219.038616 6648566.855882).
Grid cell size = (9.999946 -9.998732).
Source point count = 8120576.
Algorithm name: "i

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/SWOT_epsg2154_20250522T221758_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     5,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/sig0/SWOT_epsg2154_20250522T221758_sig0.tif
File Size: 6495x5231x1
Pixel Size: 9.999946 x -9.998732
UL:(461269.391220,6700870.222726)   LR:(526219.038616,6648566.855882)
Copy 0,0,6495,5231 to 0,0,6495,5231.

Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/coherent_power/SWOT_epsg2154_20250522T221758_coherent_power.tif
File Size: 6495x5231x1
Pixel Size: 9.999946 x -9.998732
UL:(461269.391220,6700870.222726)   LR:(526219.038616,6648566.855882)
Copy 0,0,6495,5231 to 0,0,6495,5231.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/gamma_tot/SWOT_epsg2154_20250522T221758_gamma_tot.tif
File Size: 6495x5231x1
Pixel Size: 9.9

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/SWOT_epsg2154_20250612T190303_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     5,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/sig0/SWOT_epsg2154_20250612T190303_sig0.tif
File Size: 6495x5231x1
Pixel Size: 9.999946 x -9.998732
UL:(461269.391220,6700870.222726)   LR:(526219.038616,6648566.855882)
Copy 0,0,6495,5231 to 0,0,6495,5231.

Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/coherent_power/SWOT_epsg2154_20250612T190303_coherent_power.tif
File Size: 6495x5231x1
Pixel Size: 9.999946 x -9.998732
UL:(461269.391220,6700870.222726)   LR:(526219.038616,6648566.855882)
Copy 0,0,6495,5231 to 0,0,6495,5231.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/gamma_tot/SWOT_epsg2154_20250612T190303_gamma_tot.tif
File Size: 6495x5231x1
Pixel Size: 9.9

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/SWOT_epsg2154_20260218T040347_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     5,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/sig0/SWOT_epsg2154_20260218T040347_sig0.tif
File Size: 6495x5231x1
Pixel Size: 9.999946 x -9.998732
UL:(461269.391220,6700870.222726)   LR:(526219.038616,6648566.855882)
Copy 0,0,6495,5231 to 0,0,6495,5231.

Processing file     2 of     5, 20.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/coherent_power/SWOT_epsg2154_20260218T040347_coherent_power.tif
File Size: 6495x5231x1
Pixel Size: 9.999946 x -9.998732
UL:(461269.391220,6700870.222726)   LR:(526219.038616,6648566.855882)
Copy 0,0,6495,5231 to 0,0,6495,5231.

Processing file     3 of     5, 40.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/rasters/gamma_tot/SWOT_epsg2154_20260218T040347_gamma_tot.tif
File Size: 6495x5231x1
Pixel Size: 9.9

If needed, we can put extra rasters to the same resolution and extent as the SWOT data. This is useful to compare the SWOT data with other data.

It uses gdal to rasterize the data. gdal is used as command line using os.system.

In [6]:
raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/WC_cut_merged.tif")
raster_crs = 2154
interp = 'nearest'
swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)


>>> Converting the AUXILARY rasters to tiff
gdalwarp -s_srs EPSG:2154 -t_srs EPSG:2154 -te 461269.39122028934 6648566.855881953 526219.0386155185 6700870.222726153 -ts 6495.0 5231.0 -r nearest -of GTiff /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/WC_cut_merged.tif /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/WC_cut_merged_nrow5231_ncol6495.tif
Copying color table from /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/WC_cut_merged.tif to new file.
Creating output file that is 6495P x 5231L.
Processing /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Chinon_T2D/aux_data/WC_cut_merged.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.


You can check other notebooks for more information about the library.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
file1 = "/data/scratch/globc/bonassies/data/SWOT/Chinon/SWOT_L2_HR_PIXC_013_113_237L_20240331T151613_20240331T151624_PIC0_01.nc"
file2 = "/data/scratch/globc/bonassies/data/SWOT/PortoAlegre/SWOT_L2_HR_PIXC_014_533_102R_20240506T114633_20240506T114644_PIC0_01.nc"
file3 = "/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_028_481_220L_20250220T180206_20250220T180217_PIC2_01.nc"
file4 = "/data/scratch/globc/bonassies/data/SWOT/EMSR_692/SWOT_L2_HR_PIXC_003_402_085L_20230915T070816_20230915T070827_PGC0_01.nc"

im1 = xr.open_dataset(file1)
im2 = xr.open_dataset(file2)
im3 = xr.open_dataset(file3)
im4 = xr.open_dataset(file4)

In [ ]:
im3